# Plotly OFTW App Building Challenge #

## 1. Import Necessary Libraries ##

This section imports all required libraries:
- `os` and `time` for system operations and delay handling.
- `pandas` for data manipulation and `plotly.express` for creating interactive visualizations.
- `datetime` for managing date and time objects.
- `Dash` components (from Dash, dcc, html, Input, Output, State, callback, no_update) for building the interactive dashboard.
- `dash_ag_grid` for rendering interactive data tables.
- `google.generativeai` for integrating with Google’s Generative AI API used to generate textual insights.

## 2. HELPER FUNCTIONS ##

Several helper functions are defined to support common tasks:
- `safe_to_datetime(series)`: Converts a Pandas Series containing date strings into datetime objects, returning NaT on errors.
- `create_empty_figure(title)`: Creates an empty Plotly figure with a specified title and applies a dark template.
- `create_kpi_box(text, box_id)`: Generates a styled Dash HTML Div to display a KPI value, with a specific ID used for later callback updates.

## 3. INITIALIZE DASH APP ##
The Dash application is instantiated using:

`app = Dash(__name__)`

This sets up the app for further layout configuration and interactivity.

## 4. DATA LOADING, CLEANING, AND PREPROCESSING ##

Data is loaded from two remote JSON sources:
- **Pledges data** is loaded from a given URL into a DataFrame (`df_pledges`).
- **Payments data** is loaded similarly into `df_payments`.

Both DataFrames have duplicates removed. In the payments DataFrame, rows with missing `pledge_id` values are dropped. Date columns (e.g., `pledge_date` and `payment_date`) are converted into datetime objects using the `safe_to_datetim`e helper function.

## 5. LOADING METADATA FILE ##

A metadata file is retrieved from a Google Sheets URL (exported as CSV) into a DataFrame (`df_metadata`). This file may provide supplemental information for the dashboard and is later shown in a table.

## 6. DATA MERGING AND FEATURE ENGINEERING ##

Pledges and payments are merged on the `pledge_id` column using an **outer join** to ensure all records are preserved.
- The code then renames columns to remove default suffixes (for example, `donor_id_x` becomes `donor_id` and `payment_platform_y` is renamed to `payment_platform_payment`), standardizing the names across datasets.
- A new feature, `payment_delay`, is computed as the difference (in days) between `payment_date` and `pledge_date`.
- Additionally, the year and month are extracted from the payment date for later use.

## 7. AG GRID SETUP ##

Two interactive AG Grid tables are set up:
- One grid displays the merged dataset using a selected set of display columns (e.g., `pledge_id`, `pledge_created_at`, `date`, `payment_platform`, `amount`, `payment_delay`, etc.).
- Another grid displays the metadata from the CSV file.

The display columns are controlled by a list (`final_display_columns`).

## 8. COMPUTE OFTW METRICS ##
This function calculates key metrics from the data:
- It computes the total money moved (summing the payment amounts) and a counterfactual metric.
- For pledges, metrics like Active ARR (Annualized Run Rate) are calculated from contribution amounts.
- Pledge attrition rate, counts of active donors and active pledges, and chapter-level run rates are also derived.
The function returns a dictionary containing these metrics.

## 9. UPDATED REAL LLM INSIGHTS FUNCTION USING NEW GENERATIVE MODEL API ##

This function integrates with **Google’s Generative AI**:
- It uses a generative model (e.g., `gemini-1.5-flash`) to process the metrics (provided in **Markdown**) and generate actionable insights.
- A status message (“Analyzing…”) is prepended to the output.

It includes a retry mechanism with exponential backoff, returning a fallback message if all attempts fail.

## 10. DASH APP LAYOUT ##

The layout of the dashboard is defined using Dash components:
- A header Markdown component describes the app.
- The layout includes filter components: a dropdown for selecting payment platforms and a date picker for the payment date range.
- KPI boxes (created via the `create_kpi_box` function) display key performance metrics.
- A Markdown component shows the overall metrics summary.
- A button triggers the LLM insights generation (with its output displayed in a loading component).
- Various Plotly graphs (histogram, time series, and pie chart) and the AG Grid tables for merged data and metadata are displayed.

## 11. CALLBACK FOR UPDATING VISUALIZATIONS, KPI, AND DATA TABLE ##

This callback function updates the graphs, KPI boxes, and data table based on user inputs (selected platforms and date range):
- The `filter_data` function is used to produce a filtered DataFrame.
- Visualizations (histogram, time series, pie chart) are updated using Plotly based on the filtered data.
- KPIs are recalculated (including handling for cases with fewer than five records, appending a warning message).
- The updated table data is then passed to the AG Grid.

## 12. CALLBACK FOR UPDATING OFTW METRICS SUMMARY ##

This callback recomputes the overall metrics (ignoring filters) whenever filter inputs change.
- It calls the `compute_of_tw_metrics` function, then formats the resulting metrics dictionary into a Markdown string using `format_metrics_as_markdown`.
- The Markdown output is then displayed on the dashboard.

## 13. CALLBACK FOR REAL LLM INSIGHTS USING NEW GENERATIVE MODEL API ##

Triggered when the “Get LLM Insights” button is clicked, this callback:
- Uses State to read the current metrics summary without triggering on every update.
- Calls the `get_llm_insights` function with the metrics summary, which returns generated actionable insights.
- The generated text is then displayed in the dashboard.

## 14. RUN THE APP ##

Finally, the script checks if it is run as the main module and starts the Dash development server

### 1. Import necessary libraries ###

In [3]:
import os
import time
import pandas as pd
import plotly.express as px
from datetime import datetime

from dash import Dash, dcc, html, Input, Output, State, callback, no_update
import dash_ag_grid as dag

import google.generativeai as genai

https://aistudio.google.com/app/u/1/apikey

In [5]:
GOOGLE_API_KEY = "GOOGLE_API_KEY"
if not GOOGLE_API_KEY:
    raise Exception("API KEY for Google Generative AI is not configured!")
    
client = genai.configure(api_key=GOOGLE_API_KEY)

### 2. HELPER FUNCTIONS ###

In [7]:
def safe_to_datetime(series: pd.Series) -> pd.Series:
    """
    Converts a Series of date strings to datetime, returning NaT on errors.
    
    :param series: pandas.Series with date strings
    :return: pandas.Series with datetime objects
    """
    return pd.to_datetime(series, errors='coerce')

In [8]:
def create_empty_figure(title: str='No data available') -> px.scatter:
    """
    Creates an empty figure with a given title.
    
    :param title: Figure title
    :return: Plotly figure
    """
    return px.scatter(title=title, template='plotly_dark')

In [9]:
def create_kpi_box(text: str, box_id: str) -> html.Div:
    """
    Creates a KPI box with the given text and assigns an ID.
    
    :param text: KPI text
    :param box_id: The ID for the KPI box
    :return: html.Div containing the KPI with the specified ID
    """
    return html.Div(
        text,
        id=box_id,
        style={
            'flex': '1',
            'padding': '10px',
            'color': 'white',
            'textAlign': 'center',
            'border': '1px solid #444'
        }
    )

### 3. INITIALIZE DASH APP ###

In [11]:
app = Dash(__name__)

### 4. DATA LOADING, CLEANING, AND PREPROCESSING ###

In [13]:
# URLs for the "pledges" and "payments" datasets
pledge_url = "https://storage.googleapis.com/plotly-app-challenge/one-for-the-world-pledges.json"
payment_url = "https://storage.googleapis.com/plotly-app-challenge/one-for-the-world-payments.json"

In [14]:
# Load JSON data into pandas DataFrames
try:
    df_pledges = pd.read_json(pledge_url)
except Exception as e:
    raise Exception(f"Error loading pledges: {e}")

try:
    df_payments = pd.read_json(payment_url)
except Exception as e:
    raise Exception(f"Error loading payments: {e}")

In [15]:
# Remove duplicates from both datasets
df_pledges.drop_duplicates(inplace=True)
df_payments.drop_duplicates(inplace=True)

In [16]:
df_payments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 55122 entries, 0 to 55346
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 55122 non-null  object        
 1   donor_id           55122 non-null  object        
 2   payment_platform   55122 non-null  object        
 3   portfolio          55122 non-null  object        
 4   amount             55122 non-null  float64       
 5   currency           55122 non-null  object        
 6   date               55122 non-null  datetime64[ns]
 7   counterfactuality  55122 non-null  float64       
 8   pledge_id          52470 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(6)
memory usage: 4.2+ MB


In [17]:
df_pledges.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11861 entries, 0 to 11910
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   donor_id             11861 non-null  object        
 1   pledge_id            11861 non-null  object        
 2   donor_chapter        11861 non-null  object        
 3   chapter_type         11861 non-null  object        
 4   pledge_status        11861 non-null  object        
 5   pledge_created_at    11861 non-null  datetime64[ns]
 6   pledge_starts_at     11861 non-null  datetime64[ns]
 7   pledge_ended_at      5139 non-null   datetime64[ns]
 8   contribution_amount  11861 non-null  float64       
 9   currency             11861 non-null  object        
 10  frequency            11861 non-null  object        
 11  payment_platform     11861 non-null  object        
dtypes: datetime64[ns](3), float64(1), object(8)
memory usage: 1.2+ MB


In [18]:
df_payments.head()

,id,donor_id,payment_platform,portfolio,amount,currency,date,counterfactuality,pledge_id
0,6d99138e-3f1b-49db-944a-9d13c6733ea4,,Squarespace,OFTW Top Picks,0.00,USD,2025-01-11,0.758170,NaN
1,264TJ405CZ,,Benevity,OFTW Top Picks,0.00,USD,2020-09-27,0.313636,NaN
2,5B5XGD4RTX,,Benevity,OFTW Top Picks,-100.00,USD,2024-02-01,0.313636,7373fca9-78cc-4954-baaa-b2acebac595b
3,3Z5NNTTCSD,,Benevity,OFTW Top Picks,-1.01,USD,2022-09-23,0.313636,5e6bc702-7990-4351-a5b5-4cf39401c379
4,31V9VD2AAV,,Benevity,OFTW Top Picks,0.00,USD,2021-11-19,0.313636,8a8c1bc3-a6a4-4819-a8c1-732e7a88408b


In [19]:
df_pledges.head()

,donor_id,pledge_id,donor_chapter,chapter_type,pledge_status,pledge_created_at,pledge_starts_at,pledge_ended_at,contribution_amount,currency,frequency,payment_platform
0,,472fa263-84a2-44d2-adc5-d8983fa2b560,,,Active donor,1997-01-01,1997-01-01,NaT,250.0,USD,One-Time,Givewell
1,,fa0e5a1b-ab1d-4aba-af20-2b9d6aa23312,,,One-Time,2016-12-24,2016-12-24,NaT,4000.0,USD,One-Time,Fidelity DAF
2,,71bfd12a-7877-4e21-a347-7646f823149f,,,One-Time,2016-12-24,2016-12-24,NaT,1250.0,USD,One-Time,Fidelity DAF
3,,f8811c05-5dfd-4760-b730-8532beb2f34d,,,One-Time,2016-12-24,2016-12-24,NaT,1250.0,USD,One-Time,Fidelity DAF
4,,58bf5aec-b09c-4a43-ba9d-6197c1bfd7ef,,,One-Time,2016-12-24,2016-12-24,NaT,4000.0,USD,One-Time,Fidelity DAF


In [20]:
if 'pledge_id' in df_payments.columns:
    num_missimg = df_payments['pledge_id'].isna().sum()
    if num_missimg > 0:
        df_payments = df_payments.dropna(subset=['pledge_id'])

print("Pledges DataFrame columns:", df_pledges.columns)
print("Payments DataFrame columns:", df_payments.columns)

Pledges DataFrame columns: Index(['donor_id', 'pledge_id', 'donor_chapter', 'chapter_type',
       'pledge_status', 'pledge_created_at', 'pledge_starts_at',
       'pledge_ended_at', 'contribution_amount', 'currency', 'frequency',
       'payment_platform'],
      dtype='object')
Payments DataFrame columns: Index(['id', 'donor_id', 'payment_platform', 'portfolio', 'amount', 'currency',
       'date', 'counterfactuality', 'pledge_id'],
      dtype='object')


In [21]:
# Convert date columns using safe_to_datetime
if 'pledge_date' in df_pledges.columns:
    df_pledges['pledge_date'] = safe_to_datetime(df_pledges['pledge_date'])
if 'payment_date' in df_payments.columns:
    df_payments['payment_date'] = safe_to_datetime(df_payments['payment_date'])

### 5. LOADING METADATA FILE ###

In [23]:
metadata_url = (
    "https://docs.google.com/spreadsheets/d/1XSlvYfBxqAPvdXLCG7hnVzjCCjAtQ5CpGUebhU-8sPg/"
    "export?format=csv&id=1XSlvYfBxqAPvdXLCG7hnVzjCCjAtQ5CpGUebhU-8sPg&gid=0"
)
try:
    df_metadata = pd.read_csv(metadata_url)
except Exception as e:
    raise Exception(f"Error loading metadata: {e}")
print("Metadata DataFrame columns:", df_metadata.columns)

Metadata DataFrame columns: Index(['Dataset', 'Column', 'Description', 'Notes'], dtype='object')


### 6. DATA MERGING AND FEATURE ENGINEERING ###

In [25]:
# Preserve the full merged dataset for filtering and analysis
merged_df = pd.merge(df_pledges, df_payments, on='pledge_id', how='outer')
merged_df = merged_df.dropna(subset=['pledge_id'])

In [26]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58474 entries, 0 to 58473
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   donor_id_x           58469 non-null  object        
 1   pledge_id            58474 non-null  object        
 2   donor_chapter        58469 non-null  object        
 3   chapter_type         58469 non-null  object        
 4   pledge_status        58469 non-null  object        
 5   pledge_created_at    58469 non-null  datetime64[ns]
 6   pledge_starts_at     58469 non-null  datetime64[ns]
 7   pledge_ended_at      24849 non-null  datetime64[ns]
 8   contribution_amount  58469 non-null  float64       
 9   currency_x           58469 non-null  object        
 10  frequency            58469 non-null  object        
 11  payment_platform_x   58469 non-null  object        
 12  id                   52955 non-null  object        
 13  donor_id_y           52955 non-

In [27]:
# Rename columns to remove default suffixes and standardize names.
merged_df = merged_df.rename(columns={
    'donor_id_x': 'donor_id',
    'payment_platform_x': 'payment_platform_pledge',
    'currency_x': 'currency_pledge',
    'donor_id_y': 'donor_id_payment',
    'payment_platform_y': 'payment_platform_payment',
    'currency_y': 'currency_payment'
})

In [28]:
merged_df.head()

,donor_id,pledge_id,donor_chapter,chapter_type,pledge_status,pledge_created_at,pledge_starts_at,pledge_ended_at,contribution_amount,currency_pledge,frequency,payment_platform_pledge,id,donor_id_payment,payment_platform_payment,portfolio,amount,currency_payment,date,counterfactuality
0,a4939786-bac5-4425-bf8e-15d059af7b25,000d0331-481c-4131-9eaa-e0faa91700b4,Accenture,Corporate,Active donor,2020-11-03,2024-08-02,NaT,20.0,USD,Monthly,Donational,c629d052-d383-4820-9c15-fb756fcd52dd,a4939786-bac5-4425-bf8e-15d059af7b25,Donational,Custom Portfolio,20.0,USD,2024-11-15,0.0
1,a4939786-bac5-4425-bf8e-15d059af7b25,000d0331-481c-4131-9eaa-e0faa91700b4,Accenture,Corporate,Active donor,2020-11-03,2024-08-02,NaT,20.0,USD,Monthly,Donational,b42a649a-4d9c-45eb-9785-ebc582e04ea2,a4939786-bac5-4425-bf8e-15d059af7b25,Donational,Custom Portfolio,20.0,USD,2024-09-15,0.0
2,a4939786-bac5-4425-bf8e-15d059af7b25,000d0331-481c-4131-9eaa-e0faa91700b4,Accenture,Corporate,Active donor,2020-11-03,2024-08-02,NaT,20.0,USD,Monthly,Donational,c9d4a9b0-d40b-4211-a94e-4fc180afb9b8,a4939786-bac5-4425-bf8e-15d059af7b25,Donational,Custom Portfolio,20.0,USD,2024-10-15,0.0
3,a4939786-bac5-4425-bf8e-15d059af7b25,000d0331-481c-4131-9eaa-e0faa91700b4,Accenture,Corporate,Active donor,2020-11-03,2024-08-02,NaT,20.0,USD,Monthly,Donational,e6919705-862f-4894-acaf-d20e9d9b09e0,a4939786-bac5-4425-bf8e-15d059af7b25,Donational,Custom Portfolio,20.0,USD,2024-08-15,0.0
4,a4939786-bac5-4425-bf8e-15d059af7b25,000d0331-481c-4131-9eaa-e0faa91700b4,Accenture,Corporate,Active donor,2020-11-03,2024-08-02,NaT,20.0,USD,Monthly,Donational,ce205560-609e-4fe6-94e3-fcdbf131150c,a4939786-bac5-4425-bf8e-15d059af7b25,Donational,Custom Portfolio,20.0,USD,2024-12-15,0.0


In [29]:
# Create a payment delay feature: difference between payment date and pledge creation date
if 'pledge_date' in merged_df.columns and 'payment_date' in merged_df.columns:
    merged_df['payment_delay'] = (merged_df['payment_date'] - merged_df['pledge_date']).dt.days

# Extract year and month from payment date
if 'date' in merged_df.columns:
    merged_df['payment_year'] = merged_df['date'].dt.year
    merged_df['payment_month'] = merged_df['date'].dt.year

In [30]:
# Define display columns for AG Grid (to control visible columns)
display_columns = [
    'pledge_id', 'pledge_created_at', 'date', 'payment_platform',
    'amount', 'payment_delay', 'payment_year', 'payment_month'
]
final_display_columns = display_columns

Note on NaN handling:
- For sum(), NaN values are ignored.
- For mean(), skipna=True is applied by default, so NaN are not considered.

Here we add explicit logic: when computing the mean, only non-NaN values are considered.
Real zeros (0) are treated as valid payments.

In [32]:
if merged_df['date'].dropna().empty:
    min_date, max_date = None, None
else:
    min_date = merged_df['date'].min().date()
    max_date = merged_df['date'].max().date()

if 'payment_platform' in merged_df.columns and not merged_df['payment_platform'].dropna().empty:
    fig_hist = px.histogram(
        merged_df,
        x='payment_platform',
        title='Payments by Platform',
        template='plotly_dark'
    )
    fig_hist.update_layout(bargap=0.2)
else:
    fig_hist = create_empty_figure('Payments by Platform')

if 'amount' in merged_df.columns and not merged_df['amount'].dropna().empty:
    df_time_series = merged_df.groupby('date', dropna=True)['amount'].sum().reset_index()
    fig_time = px.line(
        df_time_series,
        x='date',
        y='amount',
        title='Total Payment Amount Over Time',
        template='plotly_dark'
    )
else:
    fig_time = create_empty_figure('Total Payment Amount Over Time')

if 'amount' in merged_df.columns and not merged_df['amount'].dropna().empty:
    df_pie_init = merged_df.groupby('payment_platform_payment', dropna=True)['amount'].sum().reset_index()
    fig_pie = px.pie(
        df_pie_init,
        values='amount',
        names='payment_platform_payment',
        title="Payment Amount Distribution by Platform",
        template='plotly_dark'
    )
else:
    fig_time = create_empty_figure('"Payment Amount Distribution by Platform')

In [33]:
# For KPI calculations, use payments with valid payment dates.
payments_df = merged_df[merged_df['date'].notnull()]
total_pledges_init = merged_df['pledge_id'].nunique() if 'pledge_id' in merged_df.columns else 'N/A'
total_payments_init = payments_df.shape[0]
total_payment_amount_init = payments_df['amount'].sum() if 'amount' in payments_df.columns else 0
avg_payment_delay_init = payments_df['payment_delay'].mean() if ('payment_delay' in payments_df.columns and not payments_df['payment_delay'].dropna().empty) else None

In [34]:
# Explicit check: only consider non-NaN amount values when computing the mean
avg_payment_amount_init = payments_df['amount'][payments_df['amount'].notna()].mean() if ('amount' in payments_df.columns) else None

In [35]:
kpi_total_pledges_init = f"**Total Pledges:** {total_pledges_init}"
kpi_total_payments_init = f"**Total Payments:** {total_payments_init}"
kpi_total_payment_amount_init = f"**Total Payment Amount:** ${total_payment_amount_init:,.2f}"
kpi_avg_payment_delay_init = f"**Average Payment Delay (days):** {avg_payment_delay_init:.1f}" if avg_payment_delay_init is not None else "**Average Payment Delay:** N/A"
kpi_avg_payment_amount_init = f"**Average Payment Amount:** ${avg_payment_amount_init:,.2f}" if avg_payment_amount_init is not None else "**Average Payment Amount:** N/A"

### 7. AG GRID SETUP ###

In [37]:
grid = dag.AgGrid(
    id='payments-table',
    rowData=merged_df.to_dict('records'),
    columnDefs=[{'field': col, 'filter': True, 'sortable': True} for col in final_display_columns],
    dashGridOptions={'pagination':True}
)

In [38]:
metadata_grid = dag.AgGrid(
    id='metadata-table',
    rowData=df_metadata.to_dict('records'),
    columnDefs=[{'field': col, 'filter': True, 'sortable': True} for col in df_metadata.columns],
    dashGridOptions={'pagination':True}
)

In [39]:
def filter_data(selected_platforms: list, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Filters merged_df based on the selected platforms and date range.
    
    :param selected_platforms: List of selected platforms (payment_platform_payment)
    :param start_date: Start date in 'YYYY-MM-DD' format
    :param end_date: End date in 'YYYY-MM-DD' format
    :return: Filtered DataFrame
    """
    df = merged_df.copy()

    # For optimization, consider using pandas.query
    if selected_platforms and 'payment_platform_payment' in df.columns:
        df = df[df['payment_platform_payment'].isin(selected_platforms)]
    if start_date and end_date and 'date' in df.columns:
        df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
    
    return df

### 8. COMPUTE OFTW METRICS ###

In [41]:
def compute_of_tw_metrics() -> dict:
    """
    Computes key OFTW metrics.
    
    :return: Dictionary of metrics
    """
    metrics = {}
    payments_valid = df_payments.copy()
    if 'portfolio' in payments_valid.columns:
        payments_valid = payments_valid[~payments_valid['portfolio'].isin([
            'One for the World Discretionary Fund', 'One for the World Operating Costs'
        ])]
    # Use 'amount' as payment amount from payments.
    metrics['Money Moved (Total)'] = payments_valid['amount'].sum() if 'amount' in payments_valid.columns else 0
    if 'counterfactuality' in payments_valid.columns and 'amount' in payments_valid.columns:
        metrics['Counterfactual MM'] = (payments_valid['amount'] * payments_valid['counterfactuality']).sum()
    else:
        metrics['Counterfactual MM'] = 0

    # For pledges, use 'contribution_amount' as pledge amount and 'pledge_created_at' as pledge date.
    if 'pledge_status' in df_pledges.columns and 'contribution_amount' in df_pledges.columns:
        active_pledges = df_pledges[df_pledges['pledge_status'] == 'Active donor']
        active_arr_monthly = active_pledges['contribution_amount'].sum()
        metrics['Active ARR (Monthly)'] = active_arr_monthly
        metrics['Active ARR (Annualized)'] = active_arr_monthly * 12
    else:
        metrics['Active ARR (Monthly)'] = metrics['Active ARR (Annualized)'] = 0

    if 'pledge_status' in df_pledges.columns:
        total_pledges_count = len(df_pledges)
        cancelled = df_pledges[df_pledges['pledge_status'].isin(['Payment failure', 'Churned donor'])]
        metrics['Pledge Attrition Rate (%)'] = (len(cancelled) / total_pledges_count * 100) if total_pledges_count > 0 else None
    else:
        metrics['Pledge Attrition Rate (%)'] = None

    if 'pledge_status' in df_pledges.columns and 'donor_id' in df_pledges.columns:
        metrics['Total Active Donors'] = df_pledges[df_pledges['pledge_status'].isin(['one-time donor', 'Active donor'])]['donor_id'].nunique()
    else:
        metrics['Total Active Donors'] = None

    if 'pledge_status' in df_pledges.columns and 'donor_id' in df_pledges.columns:
        metrics['Total Active Pledges'] = df_pledges[df_pledges['pledge_status'] == 'Active donor']['donor_id'].nunique()
    else:
        metrics['Total Active Pledges'] = None

    # Use 'donor_chapter' as chapter and 'chapter_type' remains the same.
    if all(col in df_pledges.columns for col in ['donor_chapter', 'chapter_type', 'contribution_amount']):
        chapter_arr_total = df_pledges.groupby(['donor_chapter', 'chapter_type'])['contribution_amount'].sum().sum()
        metrics['Chapter ARR (Total)'] = chapter_arr_total
    else:
        metrics['Chapter ARR (Total)'] = None

    if 'region' in payments_valid.columns:
        region_money = payments_valid.groupby('region')['amount'].sum().to_dict()
        metrics['Money Moved by Region'] = region_money
    else:
        metrics['Money Moved by Region'] = "N/A"

    if 'donor_type' in df_pledges.columns and 'pledge_status' in df_pledges.columns:
        donor_type_counts = (df_pledges[df_pledges['pledge_status'] == 'Active donor']
                             .groupby('donor_type')['donor_id']
                             .nunique()
                             .to_dict() if 'donor_id' in df_pledges.columns else {})
        metrics['Active Donors by Type'] = donor_type_counts
    else:
        metrics['Active Donors by Type'] = "N/A"

    if 'pledge_status' in df_pledges.columns and 'pledge_created_at' in df_pledges.columns:
        all_pledges = df_pledges[df_pledges['pledge_status'].isin(['Pledged donor', 'Active donor'])]
        future_pledges = df_pledges[df_pledges['pledge_status'] == 'Pledged donor']
        active_pledges_count = df_pledges[df_pledges['pledge_status'] == 'Active donor']
        metrics['All Pledges (Monthly Count)'] = all_pledges.groupby(df_pledges['pledge_created_at'].dt.to_period('M')).size().mean()
        metrics['Future Pledges (Monthly Count)'] = future_pledges.groupby(df_pledges['pledge_created_at'].dt.to_period('M')).size().mean()
        metrics['Active Pledges (Monthly Count)'] = active_pledges_count.groupby(df_pledges['pledge_created_at'].dt.to_period('M')).size().mean()
        future_arr = future_pledges['contribution_amount'].sum() * 12
        active_arr = active_pledges_count['contribution_amount'].sum() * 12
        metrics['ALL ARR (Annualized)'] = future_arr + active_arr
        metrics['Future ARR (Annualized)'] = future_arr
        metrics['Active ARR (Annualized)'] = active_arr
    else:
        metrics['All Pledges (Monthly Count)'] = metrics['Future Pledges (Monthly Count)'] = metrics['Active Pledges (Monthly Count)'] = None
        metrics['ALL ARR (Annualized)'] = metrics['Future ARR (Annualized)'] = metrics['Active ARR (Annualized)'] = 0

    return metrics

In [42]:
def format_metrics_as_markdown(metrics: dict) -> str:
    """
    Formats the metrics dictionary into a markdown string with improved formatting.
    
    :param metrics: Dictionary of metrics
    :return: Formatted markdown string with key metrics in bold
    """
    md_lines = ["### **OFTW Metrics Summary**\n"]
    for key, value in metrics.items():
        md_lines.append(f"- **{key}:** {value}")
    return "\n".join(md_lines)

### 9. UPDATED REAL LLM INSIGHTS FUNCTION USING NEW GENERATIVE MODEL API ###

In [44]:
def get_llm_insights(metrics_text: str) -> str:
    """
    Generates insights using Google Generative AI.
    Adds a status "Analyzing..." before calling the LLM to inform the user.
    
    :param metrics_text: Metrics text in markdown
    :return: Generated insights or a fallback message
    """
    retries = 3
    delay = 1
    model_name = 'gemini-1.5-flash'

    # Prepend status message
    status_prefix = 'Analyzing...\n'

    for attempt in range(retries):
        try:
            model = genai.GenerativeModel(model_name)
            response = model.generate_content(
                f'You are an expert data analyst for OFTW. Analyze the following metrics and provide actionable insights: {metrics_text}'
            )
            
            if hasattr(response, 'text') and response.text and response.text.strip():
                return status_prefix + response.text.strip()
            else:
                raise ValueError('Empty or blocked response received')
        except Exception as e:
            print(f'Attempt {attempt+1} failed with error: {e}')
            print(f'Error type: {type(e)}')
            time.sleep(delay)
            delay *= 2
    
    return 'LLM insights are not available at this time. Please try again later.'

### 10. DASH APP LAYOUT ###

In [46]:
app.layout = dcc.Loading(
    id="initial-loading",
    type="default",
    children=html.Div([
        dcc.Markdown(
            '# One For The World - Data Insights Dashboard\n'
            'This internal data app provides insights into monetary pledges and payments from 2014 to the present. '
            'Below, key OFTW metrics are computed and additional analyses are provided.',
            style={
                'color': 'white',
                'textAlign': 'center'
            }
        ),
        html.Div([
            html.Div([
                html.Label(
                    'Select Payment Platform:',
                    style={'color': 'white'}
                ),
                dcc.Dropdown(
                    id='platform-filter',
                    options=[{'label': platform, 'value': platform} 
                             for platform in sorted(merged_df['payment_platform_payment'].dropna().unique())],
                    multi=True,
                    style={'color': 'black'},
                    placeholder='Select one or more platforms'
                )
            ], style={'width': '30%', 'display': 'inline-block', 'padding': '10px'}),
            html.Div([
                html.Label(
                    'Select Payment Date Range:',
                    style={'color': 'white'}),
                dcc.DatePickerRange(
                    id='date-range',
                    min_date_allowed=min_date,
                    max_date_allowed=max_date,
                    start_date=min_date,
                    end_date=max_date
                )
            ], style={'width': '30%', 'display': 'inline-block', 'padding': '10px'})
        ]),
        html.Div([
            create_kpi_box(kpi_total_pledges_init, 'kpi-total-pledges'),
            create_kpi_box(kpi_total_payments_init, 'kpi-total-payments'),
            create_kpi_box(kpi_total_payment_amount_init, 'kpi-total-payment-amount'),
            create_kpi_box(kpi_avg_payment_delay_init, 'kpi-avg-payment-delay'),
            create_kpi_box(kpi_avg_payment_amount_init, 'kpi-avg-payment-amount')
        ], style={'display': 'flex', 'justifyContent': 'space-around', 'padding': '10px'}),
        html.Div([
            dcc.Markdown(
                id='of_tw-metrics-summary',
                style={
                    'color': 'white',
                    'textAlign': 'left',
                    'padding': '10px'
                }
            )
        ], style={'padding': '20px', 'border': '1px solid #444', 'margin': '10px'}),
        html.Div([
            html.Button(
                'Get LLM Insights',
                id='get-llm-insights',
                n_clicks=0,
                style={
                    'margin': '10px',
                    'padding': '10px'
                }
            ),
            dcc.Loading(
                id="llm-loading",
                children=[
                    dcc.Markdown(
                        id="llm-insights",
                        style={
                            'width': '100%',
                            'background': '#222',
                            'color': 'white',
                            'padding': '10px',
                            'border': '1px solid #444',
                            'min-height': '150px'
                        }
                    )
                ],
                type='default'
            )
        ], style={'textAlign': 'center'}),
        dcc.Loading(
            id="graphs-loading",
            children=[
                dcc.Graph(id='platform-fig', figure=fig_hist),
                dcc.Graph(id='time-series-fig', figure=fig_time),
                dcc.Graph(id='pie-chart-fig', figure=fig_pie)
            ]
        ),
        dcc.Loading(
            id='table-loading',
            children=[html.Div([grid], style={'padding': '20px'})]
        ),
        html.Div([
            dcc.Markdown(
                '### Metadata',
                style={'color': 'white', 'textAlign': 'center'}
            ),
            metadata_grid
        ], style={'padding': '20px', 'borderTop': '1px solid #444'})
    ], style={'backgroundColor': '#111111'})
)

### 11. CALLBACK FOR UPDATING VISUALIZATIONS, KPI, AND DATA TABLE ###

In [48]:
@app.callback(
    [Output('platform-fig', 'figure'),
     Output('time-series-fig', 'figure'),
     Output('pie-chart-fig', 'figure'),
     Output('kpi-total-pledges', 'children'),
     Output('kpi-total-payments', 'children'),
     Output('kpi-total-payment-amount', 'children'),
     Output('kpi-avg-payment-delay', 'children'),
     Output('kpi-avg-payment-amount', 'children'),
     Output('payments-table', 'rowData')],
    [Input('platform-filter', 'value'),
     Input('date-range', 'start_date'),
     Input('date-range', 'end_date')]
)
def update_dashboard(selected_platforms: list, start_date: str, end_date: str):
    """
    Filters the merged dataset based on selected platforms and date range,
    then updates the graphs, KPI boxes, and table.
    
    If the filtered data has fewer than 5 records, a warning message is appended to the KPI texts.
    
    :param selected_platforms: List of selected payment platforms
    :param start_date: Start date (YYYY-MM-DD)
    :param end_date: End date (YYYY-MM-DD)
    :return: Updated figures, KPI texts, and table data
    """
    filtered_df = filter_data(selected_platforms, start_date, end_date)
    
    warning_msg = ''
    if len(filtered_df) < 5:
        warning_msg = ' (Warning: Less than 5 records for reliable analysis)'
    
    if filtered_df.empty:
        empty_fig = create_empty_figure('No data available')
        return (empty_fig, empty_fig, empty_fig,
                'Total Pledges: N/A' + warning_msg, 'Total Payments: N/A' + warning_msg, 'Total Payment Amount: N/A' + warning_msg,
                'Average Payment Delay: N/A' + warning_msg, 'Average Payment Amount: N/A' + warning_msg, [])
    
    if 'payment_platform_payment' in filtered_df.columns and not filtered_df['payment_platform_payment'].dropna().empty:
        fig_hist_updated = px.histogram(
            filtered_df,
            x='payment_platform_payment',
            title='Payments by Platform',
            template='plotly_dark'
        )
        fig_hist_updated.update_layout(bargap=0.2)
    else:
        fig_hist_updated = create_empty_figure("Payments by Platform")
    
    if 'amount' in filtered_df.columns and not filtered_df['amount'].dropna().empty:
        df_time_series_updated = filtered_df.groupby('date', dropna=True)['amount'].sum().reset_index()
        fig_time_updated = px.line(
            df_time_series_updated,
            x='date',
            y='amount',
            title='Total Payment Amount Over Time',
            template='plotly_dark'
        )
    else:
        fig_time_updated = create_empty_figure('Total Payment Amount Over Time')
    
    if 'amount' in filtered_df.columns and not filtered_df['amount'].dropna().empty:
        df_pie = filtered_df.groupby('payment_platform_payment', dropna=True)['amount'].sum().reset_index()
        fig_pie_updated = px.pie(
            df_pie,
            values='amount',
            names='payment_platform_payment',
            title='Payment Amount Distribution by Platform',
            template='plotly_dark'
        )
    else:
        fig_pie_updated = create_empty_figure('Payment Amount Distribution by Platform')
    
    payments_filtered = filtered_df[filtered_df['date'].notnull()]
    total_pledges = filtered_df['pledge_id'].nunique() if 'pledge_id' in filtered_df.columns else 'N/A'
    total_payments = payments_filtered.shape[0]
    total_payment_amount = payments_filtered['amount'].sum() if 'amount' in payments_filtered.columns else 0
    avg_payment_delay = payments_filtered['payment_delay'].mean() if ('payment_delay' in payments_filtered.columns and not payments_filtered['payment_delay'].dropna().empty) else None
    avg_payment_amount = payments_filtered['amount'][payments_filtered['amount'].notna()].mean() if ('amount' in payments_filtered.columns) else None
    
    kpi_total_pledges = f"Total Pledges: {total_pledges}" + warning_msg
    kpi_total_payments = f"Total Payments: {total_payments}" + warning_msg
    kpi_total_payment_amount = f"Total Payment Amount: ${total_payment_amount:,.2f}" + warning_msg
    kpi_avg_payment_delay = f"Average Payment Delay (days): {avg_payment_delay:.1f}" + warning_msg if avg_payment_delay is not None else "**Average Payment Delay:** N/A" + warning_msg
    kpi_avg_payment_amount = f"Average Payment Amount: ${avg_payment_amount:,.2f}" + warning_msg if avg_payment_amount is not None else "**Average Payment Amount:** N/A" + warning_msg
    
    table_data = filtered_df.to_dict('records')
    
    return (fig_hist_updated, fig_time_updated, fig_pie_updated,
            kpi_total_pledges, kpi_total_payments, kpi_total_payment_amount,
            kpi_avg_payment_delay, kpi_avg_payment_amount, table_data)

### 12. CALLBACK FOR UPDATING OFTW METRICS SUMMARY ###

In [50]:
@app.callback(
    Output("of_tw-metrics-summary", "children"),
    [Input('platform-filter', 'value'),
     Input('date-range', 'start_date'),
     Input('date-range', 'end_date')]
)
def update_metrics_summary(selected_platforms: list, start_date: str, end_date: str) -> str:
    """
    Computes the overall OFTW metrics (ignoring filters) and formats them into markdown.
    
    :param selected_platforms: List of selected platforms (not used in computation)
    :param start_date: Start date
    :param end_date: End date
    :return: Markdown string with the metrics summary
    """
    metrics = compute_of_tw_metrics()
    return format_metrics_as_markdown(metrics)

### 13. CALLBACK FOR REAL LLM INSIGHTS USING NEW GENERATIVE MODEL API ###

In [52]:
@app.callback(
    Output('llm-insights', 'children'),
    [Input('get-llm-insights', 'n_clicks')],
    [State('of_tw-metrics-summary', 'children')],
    prevent_initial_call=True
)
def update_llm_insights(n_clicks: int, metrics_md: str) -> str:
    """
    Generates insights using LLM based on the metrics summary when the "Get LLM Insights" button is clicked.
    Uses State to read the current metrics summary, preventing unnecessary API calls.
    
    :param n_clicks: Number of button clicks
    :param metrics_md: Markdown string with the metrics summary
    :return: Generated insights or an empty string
    """
    if not n_clicks:
        return ''
    return get_llm_insights(metrics_md)

### 14. RUN THE APP ###

In [54]:
if __name__ == "__main__":
    app.run_server(debug=True)